# GRU for Classification

- **Why:** 
    1. Sequential process just as sentences
    2. Short-term and Long-term memory
    3. Solves the one-hot encoding problem in traditional NNs
    4. Solves the long-term memory problem in RNNs
    4. Parameters (ie: weights) are shared
- **Problems:**

In [1]:
import os
import sys
import torch

import numpy as np
import pandas as pd

import torch.nn as nn

from tqdm import tqdm

notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../'))

from metrics import EvaluationMetric
from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

In [2]:
pd.set_option('max_colwidth', 800)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Tutorial: Building an RNN for Sentence Classification

### Step 1: Load Data

- Load train, val, and test datasets.

### Step 2: Preprocess Data

- Tokenize each word.
- Embed each word.

### Step 3: Design RNN Architecture

- Define the RNN class forward pass layers with input-to-hidden (per step/word), hidden-to-hidden (per step/word), and hidden-to-output (not per step/word, so only after all words or full sentence).
- Once one full sentence is processed, then the hidden state resets before the next sentence.

### Step 4: Implement Training Loop

- Loop the sentences, then loop the words in each sentence.
- Pass each word embedding individually into the RNN as input.
- Set up loss function, optimizer, and train the model over multiple iterations.

### Step 5: Evaluate Model Performance

- Loop the sentences in test set, then loop the words in each sentence.
- Pass each word embedding individually into the RNN as input.
- Run inference on test data (with words as input) and calculate classification metrics.

## Load Data

In [4]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
train_dataset_path = os.path.join(base_data_path, 'classification_results/july_2026_results_2026-07-08/seed3/in_domain/spacy_large/x_y_train_set.csv')
train_df = DataProcessing.load_from_file(train_dataset_path)
train_df['Ground Truth'] = train_df['Ground Truth'].astype(int)
train_df = train_df.sample(n=100)
# train_df = train_df.loc[:1000, ]

test_dataset_path = os.path.join(base_data_path, 'classification_results/july_2026_results_2026-07-08/seed3/in_domain/spacy_large/x_y_test_set.csv')
test_df = DataProcessing.load_from_file(test_dataset_path)
test_df['Ground Truth'] = test_df['Ground Truth'].astype(int)
test_df = test_df.sample(n=100)
# test_df = test_df.loc[:1000, ]
test_df.head(3)

,Base Sentence,Ground Truth,Dataset Name,Base Sentence Embedding
447,"Partly as a result of these factors, the vast majority of participants noted that progress toward the Committee's 2 percent objective could be slower than previously expected and judged that the risk of inflation running persistently above the Committee's objective had increased.",1,news_api,[-1.06029741e-01 1.38495281e-01 -8.01709816e-02 -2.31699236e-02\n -1.19971991e-01 -1.26066729e-02 -2.92132143e-02 9.30779576e-02\n 7.73568824e-02 2.47625279e+00 -1.27662912e-01 -1.36352101e-05\n 9.46746692e-02 4.31913845e-02 6.26968453e-03 -1.09549992e-01\n -3.86647023e-02 1.12464869e+00 -2.72801042e-01 -6.22727387e-02\n -5.75182214e-02 -4.29434441e-02 -1.78594086e-02 -7.71225467e-02\n 2.65595503e-02 1.01600595e-01 -5.48106758e-03 -2.88469531e-03\n -3.01733464e-02 -2.19045430e-02 2.39710556e-03 1.07614644e-01\n -3.57131623e-02 2.95588039e-02 2.67052967e-02 -7.82733187e-02\n -5.22653498e-02 5.88172302e-02 -1.15569057e-02 -9.02638510e-02\n 7.12967664e-02 9.49762464e-02 5.58562279e-02 1.63414422e-03\n 4.93800007e-02 -8.51580724e-02 -1.05399132e-01 -4.38011214e-02\n 8....
222,The goal is significant expansion in Finland and in the northern Baltic region .,0,financial_phrasebank,[ 3.64726447e-02 7.45648369e-02 -5.39408997e-02 -1.49492836e-02\n 3.98654997e-01 -6.15808964e-02 -2.16282140e-02 1.04215071e-01\n -3.05270683e-02 2.09257746e+00 -4.38735545e-01 -5.52811362e-02\n 8.05580765e-02 1.47598218e-02 1.02853648e-01 1.79174282e-02\n 8.08272958e-02 1.15694642e+00 9.88900196e-03 -1.44990787e-01\n 4.86355722e-02 -5.94912050e-03 1.55189201e-01 -4.79855500e-02\n 1.48502848e-04 -9.55267921e-02 -3.47908512e-02 2.14727134e-01\n -2.38681912e-01 2.03424960e-01 -1.87087413e-02 2.13531461e-02\n -1.02002146e-02 1.32932812e-02 2.15607077e-01 2.60415711e-02\n 2.99413614e-02 -7.10397288e-02 -9.06337574e-02 -1.22649364e-01\n 5.65475225e-02 2.24232718e-01 -5.79420403e-02 -1.70511484e-01\n 8.02650005e-02 -5.29569201e-02 -7.71330297e-03 -7.39083684e-04\n -7....
502,"Well, they're not winning this one, the Patriots.",1,yt,[-6.94605857e-02 1.71194986e-01 -1.41763508e-01 -1.49328902e-01\n 9.12654772e-02 -1.03924833e-02 3.75048704e-02 -9.14713740e-02\n -5.18950820e-03 2.40502477e+00 -1.36313602e-01 1.28648266e-01\n 1.44567594e-01 -7.99697787e-02 -1.32661998e-01 -1.74471319e-01\n -9.34941508e-03 9.12321568e-01 -1.63610682e-01 2.46664975e-02\n 4.02708352e-03 -3.26741883e-03 1.39294758e-01 -6.73534200e-02\n -1.05551004e-01 3.00328732e-02 -2.22173020e-01 -7.70590827e-02\n 1.55726656e-01 -4.84890006e-02 2.83128545e-02 1.02750838e-01\n -4.06089984e-02 8.70813653e-02 5.27103394e-02 -8.15043822e-02\n 4.00101161e-03 -5.40120490e-02 -1.62686557e-02 -2.82440875e-02\n -8.98934230e-02 1.64173916e-01 8.90315846e-02 -6.26580194e-02\n 2.55694967e-02 -8.94083350e-04 -2.81957507e-01 -9.12375748e-03\n 2....


In [5]:
len(train_df), len(test_df)

(100, 100)

## Preprocess Data

### Tokenize each word

- Before Tokenizing: Happy New Year!
- After Tokenizing: ['Happy', 'New', 'Year', '!']

In [6]:
train_sfe = SpacyFeatureExtraction(train_df, 'Base Sentence', embedding_model_name="spacy_large")
train_tokenized_df = train_sfe.split_words_in_sentence()
train_tokenized_df.head(3)

Tokenizing sentences: 100%|██████████| 100/100 [00:00<00:00, 120.51it/s]


,Base Sentence,Word,Ground Truth,Dataset Name,Base Sentence Embedding
0,Valga Lihatoostus markets its products under the Maks & Moorits trademark .,Valga,0,financial_phrasebank,[ 1.66550800e-02 4.04300056e-02 -3.04181669e-02 -9.24885049e-02\n 1.61574498e-01 1.70298163e-02 7.37217516e-02 -3.80668342e-02\n 6.67351708e-02 1.10756075e+00 -1.92116663e-01 -5.79726584e-02\n -1.17406668e-02 -1.36494249e-01 4.99321707e-02 -8.56899992e-02\n 5.41016506e-03 8.41740847e-01 -1.64153248e-01 2.93465778e-02\n 3.79750840e-02 1.49389490e-01 -2.78475005e-02 -9.43772495e-02\n 9.97622535e-02 -1.54101580e-01 -1.46998346e-01 5.86632453e-02\n -1.15398318e-01 -7.63140023e-02 -4.77815606e-02 -2.51640007e-02\n -1.15909092e-01 8.13880786e-02 2.31875032e-02 -1.08962417e-01\n 2.62849052e-02 7.04880282e-02 3.70349847e-02 2.88841724e-02\n -8.28580856e-02 3.33485864e-02 1.25199497e-01 -9.78235006e-02\n 6.15830757e-02 -2.43363325e-02 -1.25205249e-01 -5.09880483e-02\n -3....
1,Valga Lihatoostus markets its products under the Maks & Moorits trademark .,Lihatoostus,0,financial_phrasebank,[ 1.66550800e-02 4.04300056e-02 -3.04181669e-02 -9.24885049e-02\n 1.61574498e-01 1.70298163e-02 7.37217516e-02 -3.80668342e-02\n 6.67351708e-02 1.10756075e+00 -1.92116663e-01 -5.79726584e-02\n -1.17406668e-02 -1.36494249e-01 4.99321707e-02 -8.56899992e-02\n 5.41016506e-03 8.41740847e-01 -1.64153248e-01 2.93465778e-02\n 3.79750840e-02 1.49389490e-01 -2.78475005e-02 -9.43772495e-02\n 9.97622535e-02 -1.54101580e-01 -1.46998346e-01 5.86632453e-02\n -1.15398318e-01 -7.63140023e-02 -4.77815606e-02 -2.51640007e-02\n -1.15909092e-01 8.13880786e-02 2.31875032e-02 -1.08962417e-01\n 2.62849052e-02 7.04880282e-02 3.70349847e-02 2.88841724e-02\n -8.28580856e-02 3.33485864e-02 1.25199497e-01 -9.78235006e-02\n 6.15830757e-02 -2.43363325e-02 -1.25205249e-01 -5.09880483e-02\n -3....
2,Valga Lihatoostus markets its products under the Maks & Moorits trademark .,markets,0,financial_phrasebank,[ 1.66550800e-02 4.04300056e-02 -3.04181669e-02 -9.24885049e-02\n 1.61574498e-01 1.70298163e-02 7.37217516e-02 -3.80668342e-02\n 6.67351708e-02 1.10756075e+00 -1.92116663e-01 -5.79726584e-02\n -1.17406668e-02 -1.36494249e-01 4.99321707e-02 -8.56899992e-02\n 5.41016506e-03 8.41740847e-01 -1.64153248e-01 2.93465778e-02\n 3.79750840e-02 1.49389490e-01 -2.78475005e-02 -9.43772495e-02\n 9.97622535e-02 -1.54101580e-01 -1.46998346e-01 5.86632453e-02\n -1.15398318e-01 -7.63140023e-02 -4.77815606e-02 -2.51640007e-02\n -1.15909092e-01 8.13880786e-02 2.31875032e-02 -1.08962417e-01\n 2.62849052e-02 7.04880282e-02 3.70349847e-02 2.88841724e-02\n -8.28580856e-02 3.33485864e-02 1.25199497e-01 -9.78235006e-02\n 6.15830757e-02 -2.43363325e-02 -1.25205249e-01 -5.09880483e-02\n -3....


In [7]:
test_sfe = SpacyFeatureExtraction(test_df, 'Base Sentence', embedding_model_name="spacy_large")
test_tokenized_df = test_sfe.split_words_in_sentence()
test_tokenized_df.head(3)

Tokenizing sentences: 100%|██████████| 100/100 [00:00<00:00, 146.56it/s]


,Base Sentence,Word,Ground Truth,Dataset Name,Base Sentence Embedding
0,"Partly as a result of these factors, the vast majority of participants noted that progress toward the Committee's 2 percent objective could be slower than previously expected and judged that the risk of inflation running persistently above the Committee's objective had increased.",Partly,1,news_api,[-1.06029741e-01 1.38495281e-01 -8.01709816e-02 -2.31699236e-02\n -1.19971991e-01 -1.26066729e-02 -2.92132143e-02 9.30779576e-02\n 7.73568824e-02 2.47625279e+00 -1.27662912e-01 -1.36352101e-05\n 9.46746692e-02 4.31913845e-02 6.26968453e-03 -1.09549992e-01\n -3.86647023e-02 1.12464869e+00 -2.72801042e-01 -6.22727387e-02\n -5.75182214e-02 -4.29434441e-02 -1.78594086e-02 -7.71225467e-02\n 2.65595503e-02 1.01600595e-01 -5.48106758e-03 -2.88469531e-03\n -3.01733464e-02 -2.19045430e-02 2.39710556e-03 1.07614644e-01\n -3.57131623e-02 2.95588039e-02 2.67052967e-02 -7.82733187e-02\n -5.22653498e-02 5.88172302e-02 -1.15569057e-02 -9.02638510e-02\n 7.12967664e-02 9.49762464e-02 5.58562279e-02 1.63414422e-03\n 4.93800007e-02 -8.51580724e-02 -1.05399132e-01 -4.38011214e-02\n 8....
1,"Partly as a result of these factors, the vast majority of participants noted that progress toward the Committee's 2 percent objective could be slower than previously expected and judged that the risk of inflation running persistently above the Committee's objective had increased.",as,1,news_api,[-1.06029741e-01 1.38495281e-01 -8.01709816e-02 -2.31699236e-02\n -1.19971991e-01 -1.26066729e-02 -2.92132143e-02 9.30779576e-02\n 7.73568824e-02 2.47625279e+00 -1.27662912e-01 -1.36352101e-05\n 9.46746692e-02 4.31913845e-02 6.26968453e-03 -1.09549992e-01\n -3.86647023e-02 1.12464869e+00 -2.72801042e-01 -6.22727387e-02\n -5.75182214e-02 -4.29434441e-02 -1.78594086e-02 -7.71225467e-02\n 2.65595503e-02 1.01600595e-01 -5.48106758e-03 -2.88469531e-03\n -3.01733464e-02 -2.19045430e-02 2.39710556e-03 1.07614644e-01\n -3.57131623e-02 2.95588039e-02 2.67052967e-02 -7.82733187e-02\n -5.22653498e-02 5.88172302e-02 -1.15569057e-02 -9.02638510e-02\n 7.12967664e-02 9.49762464e-02 5.58562279e-02 1.63414422e-03\n 4.93800007e-02 -8.51580724e-02 -1.05399132e-01 -4.38011214e-02\n 8....
2,"Partly as a result of these factors, the vast majority of participants noted that progress toward the Committee's 2 percent objective could be slower than previously expected and judged that the risk of inflation running persistently above the Committee's objective had increased.",a,1,news_api,[-1.06029741e-01 1.38495281e-01 -8.01709816e-02 -2.31699236e-02\n -1.19971991e-01 -1.26066729e-02 -2.92132143e-02 9.30779576e-02\n 7.73568824e-02 2.47625279e+00 -1.27662912e-01 -1.36352101e-05\n 9.46746692e-02 4.31913845e-02 6.26968453e-03 -1.09549992e-01\n -3.86647023e-02 1.12464869e+00 -2.72801042e-01 -6.22727387e-02\n -5.75182214e-02 -4.29434441e-02 -1.78594086e-02 -7.71225467e-02\n 2.65595503e-02 1.01600595e-01 -5.48106758e-03 -2.88469531e-03\n -3.01733464e-02 -2.19045430e-02 2.39710556e-03 1.07614644e-01\n -3.57131623e-02 2.95588039e-02 2.67052967e-02 -7.82733187e-02\n -5.22653498e-02 5.88172302e-02 -1.15569057e-02 -9.02638510e-02\n 7.12967664e-02 9.49762464e-02 5.58562279e-02 1.63414422e-03\n 4.93800007e-02 -8.51580724e-02 -1.05399132e-01 -4.38011214e-02\n 8....


### Embed each word

In [8]:
train_embeddings_df = train_sfe.word_embeddings_extraction(tokenized_words_with_metadata_df=train_tokenized_df, reorder_cols=["Base Sentence", "Word", "Word Embedding", "Ground Truth"])
train_embeddings_df.head(3)

Embedding words: 100%|██████████| 4708/4708 [00:07<00:00, 661.89it/s]


,Base Sentence,Word,Word Embedding,Ground Truth,Dataset Name,Base Sentence Embedding
0,Valga Lihatoostus markets its products under the Maks & Moorits trademark .,Valga,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...]",0,financial_phrasebank,[ 1.66550800e-02 4.04300056e-02 -3.04181669e-02 -9.24885049e-02\n 1.61574498e-01 1.70298163e-02 7.37217516e-02 -3.80668342e-02\n 6.67351708e-02 1.10756075e+00 -1.92116663e-01 -5.79726584e-02\n -1.17406668e-02 -1.36494249e-01 4.99321707e-02 -8.56899992e-02\n 5.41016506e-03 8.41740847e-01 -1.64153248e-01 2.93465778e-02\n 3.79750840e-02 1.49389490e-01 -2.78475005e-02 -9.43772495e-02\n 9.97622535e-02 -1.54101580e-01 -1.46998346e-01 5.86632453e-02\n -1.15398318e-01 -7.63140023e-02 -4.77815606e-02 -2.51640007e-02\n -1.15909092e-01 8.13880786e-02 2.31875032e-02 -1.08962417e-01\n 2.62849052e-02 7.04880282e-02 3.70349847e-02 2.88841724e-02\n -8.28580856e-02 3.33485864e-02 1.25199497e-01 -9.78235006e-02\n 6.15830757e-02 -2.43363325e-02 -1.25205249e-01 -5.09880483e-02\n -3....
1,Valga Lihatoostus markets its products under the Maks & Moorits trademark .,Lihatoostus,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...]",0,financial_phrasebank,[ 1.66550800e-02 4.04300056e-02 -3.04181669e-02 -9.24885049e-02\n 1.61574498e-01 1.70298163e-02 7.37217516e-02 -3.80668342e-02\n 6.67351708e-02 1.10756075e+00 -1.92116663e-01 -5.79726584e-02\n -1.17406668e-02 -1.36494249e-01 4.99321707e-02 -8.56899992e-02\n 5.41016506e-03 8.41740847e-01 -1.64153248e-01 2.93465778e-02\n 3.79750840e-02 1.49389490e-01 -2.78475005e-02 -9.43772495e-02\n 9.97622535e-02 -1.54101580e-01 -1.46998346e-01 5.86632453e-02\n -1.15398318e-01 -7.63140023e-02 -4.77815606e-02 -2.51640007e-02\n -1.15909092e-01 8.13880786e-02 2.31875032e-02 -1.08962417e-01\n 2.62849052e-02 7.04880282e-02 3.70349847e-02 2.88841724e-02\n -8.28580856e-02 3.33485864e-02 1.25199497e-01 -9.78235006e-02\n 6.15830757e-02 -2.43363325e-02 -1.25205249e-01 -5.09880483e-02\n -3....
2,Valga Lihatoostus markets its products under the Maks & Moorits trademark .,markets,"[-0.13037, 0.42151, 0.075401, 0.017214, 0.70857, -0.32458, -0.1552, 0.42138, -0.23915, 2.5459, -0.77807, -0.35454, 0.67519, -0.54534, 0.20437, -0.13132, -0.07841, 1.2377, 0.15219, 0.49324, -0.18138, 0.5122, 0.17869, -0.023629, 0.19169, 0.18584, -0.14259, 0.37657, -0.68121, -0.16993, -0.13291, -0.31561, 0.21109, -0.60136, 0.30594, 0.060208, 0.46146, 0.12301, -0.062679, 0.39109, -0.29777, -0.32101, 0.3069, -0.040102, -0.052329, 0.11883, -0.20358, -0.47376, -0.41348, -0.12139, 0.5062, 0.060497, -0.61093, 0.34281, 0.5904, -0.28708, -0.17519, -0.47845, 0.083756, -0.2272, -0.038826, -0.33876, -0.40506, 0.33692, 0.14255, -0.26704, 0.40084, -0.10776, 0.094497, 0.090037, 0.19084, 0.25273, -0.14437, 0.40455, -0.49852, -0.0087572, -0.1763, 0.20549, 0.06383, 0.18286, 0.3789, 0.15232, 0.094728, -0....",0,financial_phrasebank,[ 1.66550800e-02 4.04300056e-02 -3.04181669e-02 -9.24885049e-02\n 1.61574498e-01 1.70298163e-02 7.37217516e-02 -3.80668342e-02\n 6.67351708e-02 1.10756075e+00 -1.92116663e-01 -5.79726584e-02\n -1.17406668e-02 -1.36

In [9]:
print(f"train_embeddings_df rows: {len(train_embeddings_df)}")

train_embeddings_df rows: 4708


In [10]:
test_embeddings_df = test_sfe.word_embeddings_extraction(tokenized_words_with_metadata_df=test_tokenized_df, reorder_cols=["Base Sentence", "Word", "Word Embedding", "Ground Truth"])
test_embeddings_df.head(3)

Embedding words: 100%|██████████| 3800/3800 [00:05<00:00, 652.87it/s]


,Base Sentence,Word,Word Embedding,Ground Truth,Dataset Name,Base Sentence Embedding
0,"Partly as a result of these factors, the vast majority of participants noted that progress toward the Committee's 2 percent objective could be slower than previously expected and judged that the risk of inflation running persistently above the Committee's objective had increased.",Partly,"[0.11486, -0.10152, 0.12248, 0.074491, -0.74353, 0.14299, -0.24773, 0.18237, -0.34566, 2.5448, 0.021364, 0.027463, -0.22587, 0.16955, 0.22318, 0.016107, 0.0024367, 0.78027, 0.44525, 0.0014694, -0.21975, -0.19643, -0.12548, 0.37826, 0.142, 0.023143, -0.084489, 0.086123, -0.36227, -0.37477, -0.093234, 0.071277, -0.31179, -0.16349, -0.30843, 0.13547, 0.084076, 0.35857, 0.019227, -0.20848, 0.2484, -0.14218, 0.25056, 0.25562, -0.17696, 0.23662, 0.25369, -0.48089, 0.19875, 0.3759, 0.19821, 0.10301, -0.29112, 0.14599, -0.17365, -0.37234, -0.30125, -0.41194, -0.16553, 0.088223, -0.5056, 0.084289, -0.07237, -0.12309, -0.59301, 0.11142, -0.11875, -0.52374, 0.076249, 0.17087, -0.076031, 0.36443, -0.31231, -0.12644, 0.24268, -0.26139, 0.036232, 0.30519, -0.15154, 0.039916, -0.20338, 0.14293, -0.30...",1,news_api,[-1.06029741e-01 1.38495281e-01 -8.01709816e-02 -2.31699236e-02\n -1.19971991e-01 -1.26066729e-02 -2.92132143e-02 9.30779576e-02\n 7.73568824e-02 2.47625279e+00 -1.27662912e-01 -1.36352101e-05\n 9.46746692e-02 4.31913845e-02 6.26968453e-03 -1.09549992e-01\n -3.86647023e-02 1.12464869e+00 -2.72801042e-01 -6.22727387e-02\n -5.75182214e-02 -4.29434441e-02 -1.78594086e-02 -7.71225467e-02\n 2.65595503e-02 1.01600595e-01 -5.48106758e-03 -2.88469531e-03\n -3.01733464e-02 -2.19045430e-02 2.39710556e-03 1.07614644e-01\n -3.57131623e-02 2.95588039e-02 2.67052967e-02 -7.82733187e-02\n -5.22653498e-02 5.88172302e-02 -1.15569057e-02 -9.02638510e-02\n 7.12967664e-02 9.49762464e-02 5.58562279e-02 1.63414422e-03\n 4.93800007e-02 -8.51580724e-02 -1.05399132e-01 -4.38011214e-02\n 8....
1,"Partly as a result of these factors, the vast majority of participants noted that progress toward the Committee's 2 percent objective could be slower than previously expected and judged that the risk of inflation running persistently above the Committee's objective had increased.",as,"[-0.10648, -0.016295, -0.22755, -0.18934, 0.14167, 0.27404, -0.028969, -0.3154, -0.4032, 2.7085, 0.15321, 0.15345, -0.085798, -0.17395, -0.0059932, -0.044821, -0.027702, 1.0713, -0.31542, -0.25109, -0.31534, 0.014133, -0.38718, 0.13518, -0.067774, 0.34392, -0.2021, 0.0017084, 0.053716, -0.066964, 0.12081, 0.21121, 0.036278, 0.10395, 0.13312, -0.29282, 0.12094, 0.096785, -0.13612, -0.14153, -0.0035055, 0.23564, 0.044989, -0.0029994, -0.060647, -0.15818, -0.3098, -0.25685, -0.077777, 0.10374, -0.032998, 0.1671, -0.17314, 0.11451, 0.066825, -0.00076626, 0.021958, -0.028378, -0.14265, -0.24112, 0.11367, -0.10256, 0.023031, -0.027696, -0.069263, -0.15693, -0.026229, -0.14117, -0.050208, 0.072067, 0.13692, 0.16498, -0.089294, 0.08257, -0.12619, 0.29701, 0.1799, -0.12885, 0.15327, 0.2022, -0....",1,news_api,[-1.06029741e-01 1.38495281e-01 -8.01709816e-02 -2.31699236e-02\n -1.19971991e-01 -1.26066729e-02 -2.92132143e-02 9.30779576e-02\n 7.73568824e-02 2.47625279e+00 -1.27662912e-01 -1.36352101e-05\n 9.46746692e-02 4.31913845e-02 6.26968453e-03 -1.09549992e-01\n -3.86647023e-02 1.12464869e+00 -2.72801042e-01 -6.22727387e-02\n -5.75182214e-02 -4.29434441e-02 -1.78594086e-02 -7.71225467e-02\n 2.65595503e-02 1.01600595e-01 -5.48106758e-03 -2.88469531e-03\n -3.01733464e-02 -2.19045430e-02 2.39710556e-03 1.07614644e-01\n -3.57131623e-02 2.95588039e-02 2.67052967e-02 -7.82733187e-02\n -5.22653498e-02 5.88172302e-02 -1.15569057e-02 -9.02638510e-02\n 7.12967664e-02 9.49762464e-02 5.58562279e-02 1.63414422e-03\n 4.93800007e-02 -8.51580724e-02 -1.05399132e-01 -4.38011214e-02\n 8....
2,"Partly as a result of these factors, the vast majority of participants noted that progress toward the Committee's 2 percent objective could be slowe

In [11]:
print(f"Rows: {len(test_embeddings_df)}")
print(f"Unique sentences: {test_embeddings_df['Base Sentence'].nunique()}")
print(f"Empty Word column: {test_embeddings_df['Word'].isna().sum()}")
print(f"Zero embeddings: {(test_embeddings_df['Word Embedding'].apply(lambda x: np.all(x == 0))).sum()}")
print(test_embeddings_df[['Base Sentence', 'Word', 'Word Embedding']].head(10))

Rows: 3800
Unique sentences: 100
Empty Word column: 0
Zero embeddings: 54
                                                                                                                                                                                                                                                                              Base Sentence  \
0  Partly as a result of these factors, the vast majority of participants noted that progress toward the Committee's 2 percent objective could be slower than previously expected and judged that the risk of inflation running persistently above the Committee's objective had increased.   
1  Partly as a result of these factors, the vast majority of participants noted that progress toward the Committee's 2 percent objective could be slower than previously expected and judged that the risk of inflation running persistently above the Committee's objective had increased.   
2  Partly as a result of these factors, the vast majority of part

In [12]:
# Check original test_df
print(f"Original test_df sentences: {len(test_df)}")
print(f"Unique sentences in original: {test_df['Base Sentence'].nunique()}")

# Check after tokenization
print(f"\nAfter tokenization:")
print(f"Total rows in test_tokenized_df: {len(test_tokenized_df)}")
print(f"Unique sentences: {test_tokenized_df['Base Sentence'].nunique()}")

# Check after embedding extraction
print(f"\nAfter embedding extraction:")
print(f"Total rows in test_embeddings_df: {len(test_embeddings_df)}")
print(f"Unique sentences: {test_embeddings_df['Base Sentence'].nunique()}")
print(f"Rows with NaN embeddings: {test_embeddings_df['Word Embedding'].isna().sum()}")

# Check which sentences are missing
original_sentences = set(test_df['Base Sentence'].unique())
processed_sentences = set(test_embeddings_df['Base Sentence'].unique())
missing_sentences = original_sentences - processed_sentences
print(f"\nMissing {len(missing_sentences)} sentences after processing")



Original test_df sentences: 100
Unique sentences in original: 100

After tokenization:
Total rows in test_tokenized_df: 3800
Unique sentences: 100

After embedding extraction:
Total rows in test_embeddings_df: 3800
Unique sentences: 100
Rows with NaN embeddings: 0

Missing 0 sentences after processing


## Design GRU Architecture

### Image of RNN Architecture from USC NLP with Max

![image.png](attachment:image.png)

![image-2.png](attachment:image-2.png)

### Notes on the above

- () vs []: 
    - (-1, 1) = {x | -1 < x < 1} 
    - [-1, 1] = {x | -1 <= x <= 1} 

---

1. **Sigmoid:** Binary classification  
    - Also known as logistic regression  

2. **tanh:** Hidden state  
   - Use tanh for hidden state because it bounds and stabilizes values  
   - tanh bounds hidden state between (-1, 1)  
   - bounded hidden state keeps values stable over time and helps control gradients  
     - stability prevents values from growing or shrinking uncontrollably across time steps  
     - stability helps control gradients during backpropagation  
       - prevents exploding gradients  
         - exploding gradients occur when values become very large, causing unstable learning  
       - vanishing gradients can still occur  
         - derivatives approach 0 → gradients shrink → learning becomes very slow or stops  

   - **Analogy (Basketball Games)**  
     - tanh acts like a referee enforcing rules (bounds) on every game (time step)  
     - ensures each game follows consistent limits (values stay within range)  
     - prevents chaos (unbounded growth) across games (time steps)  

3. **ReLU:**  

4. **Softmax:** Multi-class classification  


### Code

In [ ]:
class GRU(nn.Module):
    """
    A single GRU cell, written out explicitly (no nn.GRU).

    The four equations we are implementing, from the screenshot:

        r_t  = sigma( W^r h_{t-1} + U^r x_t + b^r )        reset gate
        z_t  = sigma( W^z h_{t-1} + U^z x_t + b^z )        update gate
        h~_t = tanh( W (r_t (*) h_{t-1}) + U x_t + b )     candidate state
        h_t  = (1 - z_t) (*) h_{t-1} + z_t (*) h~_t        new state

    where (*) is the elementwise (Hadamard) product.

    Two operations show up, and confusing them is the #1 source of bugs:

      @  matmul      a (1, D) vector times a (D, H) matrix -> (1, H).
                     The shape CHANGES. This is a learned projection.
                     Used wherever a weight matrix appears.

      *  elementwise a (1, H) gate times a (1, H) state -> (1, H).
                     The shape STAYS. This is one vector scaling another,
                     entry by entry. No parameters involved -- the gate
                     already did its learning when it was produced.
                     Used wherever (*) appears above.

    Note the screenshot writes W^r h_{t-1} + U^r x_t as two separate matmuls.
    Stacking [x_t ; h_{t-1}] into one vector and using a single matrix of
    shape (D + H, H) is mathematically identical -- the top D rows play the
    role of U, the bottom H rows play the role of W -- and it is one matmul
    instead of two. That is the "concat trick" and it is what nn.GRU does.
    """

    def __init__(self, input_embedding_size, hidden_size, output_size):
        super(GRU, self).__init__()

        # Width of one word embedding, i.e. the D above.
        input_size = input_embedding_size
        self.input_embedding_size = input_size

        # Width of the hidden state, i.e. the H above.
        self.hidden_size = hidden_size

        # Three separate matrices. Each takes the concatenated [x_t ; h_{t-1}]
        # and projects it down to hidden_size.
        #
        # They MUST be three distinct Parameters. If the reset gate and the
        # update gate share a matrix they compute the identical function, so
        # r_t == z_t for every input forever, and there is no gating at all.
        #
        # Dividing by sqrt(fan_in) is Xavier-style scaling: it keeps the
        # variance of the pre-activation near 1 so sigmoid/tanh start in
        # their responsive middle region instead of saturating flat at the
        # ends (where the derivative is ~0 and nothing learns).
        self.W_hidden_reset = nn.Parameter(
            torch.randn(input_size + hidden_size, hidden_size)
            / np.sqrt(input_size + hidden_size)
        )
        self.W_hidden_update = nn.Parameter(
            torch.randn(input_size + hidden_size, hidden_size)
            / np.sqrt(input_size + hidden_size)
        )
        self.W_hidden = nn.Parameter(  # candidate matrix, the W and U of h~_t
            torch.randn(input_size + hidden_size, hidden_size)
            / np.sqrt(input_size + hidden_size)
        )

        # One bias per matrix, each of width hidden_size to match its output.
        # Biases go INSIDE the activation: sigma(x @ W + b), never
        # sigma(x @ W) + b. Adding an unbounded learned parameter after the
        # sigmoid lets the "gate" leave [0, 1], and then (1 - z_t) can go
        # negative and the state update stops being a convex mixture.
        self.b_hidden_reset = nn.Parameter(torch.zeros(hidden_size))
        self.b_hidden_update = nn.Parameter(torch.zeros(hidden_size))
        self.b_hidden = nn.Parameter(torch.zeros(hidden_size))

        # Output head: maps the hidden state to class logits.
        self.W_out = nn.Parameter(
            torch.randn(hidden_size, output_size) / np.sqrt(hidden_size)
        )
        self.b_out = nn.Parameter(torch.zeros(output_size))

    def forward(self, input_tensor, hidden_tensor):
        """
        input_tensor  x_t      (1, input_size)   this timestep's word embedding
        hidden_tensor h_{t-1}  (1, hidden_size)  everything the cell remembers

        Returns (h_t, logits). h_t gets fed back in as hidden_tensor at t+1.
        """
        x_t = input_tensor            # (1, D)
        h_prev = hidden_tensor        # (1, H)

        # ---- Gates -------------------------------------------------------
        # Both gates look at the same thing: the current word alongside the
        # full previous state.
        i_h = torch.cat((x_t, h_prev), dim=1)                    # (1, D + H)

        # r_t: "how much of the past should the candidate be allowed to see?"
        # sigmoid squashes to [0, 1]. 0 = ignore the past entirely and read
        # this word fresh; 1 = full access to the past.
        reset_t = torch.sigmoid(i_h @ self.W_hidden_reset
                                + self.b_hidden_reset)           # (1, H)

        # z_t: "how much of the state should this timestep actually rewrite?"
        # 0 = keep h_{t-1} untouched; 1 = overwrite it with the candidate.
        update_t = torch.sigmoid(i_h @ self.W_hidden_update
                                 + self.b_hidden_update)         # (1, H)

        # ---- Candidate state h~_t ----------------------------------------
        # Apply the reset gate to h_prev BEFORE concatenating. This is the
        # whole point of r_t: if we concatenated the raw h_prev and then
        # added a reset term, the ungated past would still leak in through
        # the concat and the gate could never suppress it.
        #
        # tanh bounds the candidate to [-1, 1]. Without it the state is
        # unbounded and, because h_t feeds back into itself every timestep,
        # it can grow without limit -- the exploding-state problem.
        gated = torch.cat((x_t, reset_t * h_prev), dim=1)        # (1, D + H)
        h_tilde = torch.tanh(gated @ self.W_hidden
                             + self.b_hidden)                    # (1, H)

        # ---- New state h_t -----------------------------------------------
        # A convex mixture: every unit interpolates between "keep the old
        # value" and "take the new candidate", with z_t choosing where on
        # that line to land. The weights sum to 1 per unit.
        #
        # The (1 - z_t) * h_prev term is the gradient highway, and it is the
        # entire reason GRUs beat vanilla RNNs on long sequences. Notice no
        # weight matrix and no tanh touches it. In a vanilla RNN, gradients
        # flowing backward through time get multiplied by W and by tanh'
        # (which is < 1) at every single step, so they shrink exponentially.
        # Here, if a unit sets z_t ~ 0, then d h_t / d h_{t-1} ~ 1 for that
        # unit and the gradient passes through untouched. The network can
        # LEARN to open that path for information it needs to carry far.
        h_t = (1 - update_t) * h_prev + update_t * h_tilde       # (1, H)

        # ---- Output ------------------------------------------------------
        # Project the state to output_size. Returning raw logits (no sigmoid)
        # so you can use nn.BCEWithLogitsLoss, which fuses sigmoid + BCE into
        # one numerically stable operation. Applying sigmoid here and then
        # nn.BCELoss gives the same math on paper but can underflow to log(0)
        # once the model gets confident. Call torch.sigmoid(logits) yourself
        # at eval time if you want probabilities.
        logits = h_t @ self.W_out + self.b_out                   # (1, output)

        # Return h_t, NOT h_tilde. h_tilde is scratch work -- the candidate
        # the cell considered. h_t is the gated result the cell committed to.
        # Returning h_tilde silently discards the gating across timesteps and
        # turns this back into a vanilla RNN. No error, no crash, just a
        # worse model.
        return h_t, logits

    def init_hidden(self):
        """h_0: the cell starts with no memory. Called once per sentence."""
        return torch.zeros(1, self.hidden_size)

In [ ]:
class GRU_Linear(nn.Module):
    """
    GRU using nn.Linear layers.

    Same four equations as the raw-Parameter version:

        r_t  = sigma( W^r h_{t-1} + U^r x_t + b^r )        reset gate
        z_t  = sigma( W^z h_{t-1} + U^z x_t + b^z )        update gate
        h~_t = tanh( W (r_t (*) h_{t-1}) + U x_t + b )     candidate state
        h_t  = (1 - z_t) (*) h_{t-1} + z_t (*) h~_t        new state

    What nn.Linear buys you: nn.Linear(in, out) IS `x @ W.T + b`, weight and
    bias fused into one object. Every place the raw version wrote
    `i_h @ self.W_hidden_reset + self.b_hidden_reset`, this version writes
    `self.reset_gate(i_h)`. That makes the bias-outside-the-sigmoid bug
    structurally impossible: the bias is inside the Linear, so it lands in
    the pre-activation whether you think about it or not.

    What nn.Linear does NOT buy you: the gating still has to be written by
    hand. The (*) products below are elementwise (`*`), between two (1, H)
    tensors. Only the projections -- the shape-changing (1, D+H) -> (1, H)
    maps -- go through a Linear.
    """

    def __init__(self, input_embedding_size, hidden_size, output_size):
        super(GRU_Linear, self).__init__()

        # x_t (word embedding), width D
        input_size = input_embedding_size
        self.input_embedding_size = input_size

        # h_t size, width H
        self.hidden_size = hidden_size

        # THREE separate Linear layers, not one reused three times.
        # A Linear layer owns its weight and bias; calling the same layer
        # twice runs the same function twice. Sharing one between the reset
        # and update gate would make r_t == z_t for every input, forever,
        # and the gating would do nothing.
        #
        # Each maps the concatenated [x_t ; h_{t-1}] down to hidden_size.
        # Concatenating and using one (D+H, H) map is identical to the two
        # separate W and U matmuls in the equations above -- the top D rows
        # act as U, the bottom H rows act as W.
        self.reset_gate = nn.Linear(input_size + hidden_size, hidden_size)
        self.update_gate = nn.Linear(input_size + hidden_size, hidden_size)
        self.candidate = nn.Linear(input_size + hidden_size, hidden_size)

        # h_t -> y_hat
        self.hidden_to_output = nn.Linear(hidden_size, output_size)

        # Xavier scaling keeps the variance of each pre-activation near 1, so
        # sigmoid and tanh start life in their responsive middle region rather
        # than saturated flat at the ends where the derivative is ~0 and no
        # gradient flows. Linear's default bias init is a small uniform; zero
        # is the cleaner starting point for a gate.
        for layer in (self.reset_gate, self.update_gate,
                      self.candidate, self.hidden_to_output):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, input_tensor, hidden_tensor):
        """
        x_t       = input_tensor          (1, D)
        h_{t-1}   = hidden_tensor         (1, H)
        i_h       = [x_t ; h_{t-1}]       (1, D + H)

        r_t       = sigma(Linear_reset(i_h))
        z_t       = sigma(Linear_update(i_h))
        h~_t      = tanh(Linear_cand([x_t ; r_t * h_{t-1}]))
        h_t       = (1 - z_t) * h_{t-1} + z_t * h~_t
        y_hat     = Linear_out(h_t)
        """
        x_t = input_tensor            # (1, D)
        h_prev = hidden_tensor        # (1, H)

        # 1. i_h = [x_t ; h_{t-1}]. Both gates read the current word and the
        #    full previous state.
        i_h = torch.cat((x_t, h_prev), dim=1)                    # (1, D + H)

        # 2a. r_t: how much of the past may the candidate see?
        #     sigmoid squashes to [0, 1]. 0 = read this word fresh, ignoring
        #     memory; 1 = full access to the past.
        reset_t = torch.sigmoid(self.reset_gate(i_h))            # (1, H)

        # 2b. z_t: how much of the state does this timestep rewrite?
        #     0 = keep h_{t-1} untouched; 1 = overwrite with the candidate.
        update_t = torch.sigmoid(self.update_gate(i_h))          # (1, H)

        # 3. h~_t: the candidate. Gate h_prev BEFORE concatenating -- that is
        #    the entire job of r_t. Reusing `i_h` here would smuggle an
        #    ungated copy of h_prev into the candidate through the concat,
        #    and the reset gate could never actually suppress anything.
        #
        #    tanh bounds the candidate to [-1, 1]. h_t feeds back into itself
        #    every timestep, so an unbounded candidate lets the state grow
        #    without limit.
        gated = torch.cat((x_t, reset_t * h_prev), dim=1)        # (1, D + H)
        h_tilde = torch.tanh(self.candidate(gated))              # (1, H)

        # 4. h_t: a convex mixture. Each unit interpolates between "keep the
        #    old value" and "take the new candidate"; z_t picks where on that
        #    line to land. The two weights sum to 1 per unit.
        #
        #    `(1 - z_t) * h_prev` is the gradient highway, and it is why GRUs
        #    beat vanilla RNNs on long sequences. No Linear and no tanh touch
        #    it. In a vanilla RNN, a gradient flowing backward through time is
        #    multiplied by W and by tanh' (< 1) at every step, so it shrinks
        #    exponentially. Here, if a unit learns z_t ~ 0, then
        #    d h_t / d h_{t-1} ~ 1 for that unit and the gradient passes
        #    through untouched.
        h_t = (1 - update_t) * h_prev + update_t * h_tilde       # (1, H)

        # 5. Raw logits, no sigmoid. Pair with nn.BCEWithLogitsLoss, which
        #    fuses sigmoid + BCE into one numerically stable op. Call
        #    torch.sigmoid(logits) yourself at eval time for probabilities.
        logits = self.hidden_to_output(h_t)                      # (1, output)

        # Return h_t, NOT h_tilde. h_tilde is the candidate the cell merely
        # considered; h_t is what it committed to. Handing h_tilde forward
        # discards the gating across timesteps and quietly degrades this back
        # into a vanilla RNN -- same shapes, no error, worse model.
        return h_t, logits

    def init_hidden(self):
        """h_0: the cell starts with no memory. Called once per sentence."""
        return torch.zeros(1, self.hidden_size)

In [ ]:
input_embedding = torch.tensor(train_embeddings_df['Word Embedding'][0])
hidden_size = 128

output_size = 1
rnn_classifier = RNN_Linear(input_embedding, hidden_size, output_size) # build the architecture
rnn_classifier

## Implement Training Loop

### Notes

- learning_rate ($ \alpha $)
    1. Also called step size [3]
    2. How fast to learn
- optimizer
    1. Gradient Descent (GD): reduce the squared error by calculating the partial derivative of E wrt each weight [2]
        1. simple and fundamental [3]
        2. Gradient sometimes referred to as *first-order* information of a function
        3. may not be guaranteed to arrive at even a local minimum in a reasonable amount of time, but it often finds a very low value of the cost function quickly enough to be useful [4.1].
        4. has often been regarded as slow or unreliable [4.1].
    2. Stochastic (GD): selects examples (in minibatches [4.1]) randomly from training set rather than cycling through them.
        1. faster, eﬀective for large-scale problems [3]
        2. An extension of the (GD) algo [4.1]
        3. the gradient is an expectation--may be appox estimated using a small set of samples [4.1]
        4. main way to train large linear models on very large datasets [4.1]
- loss(y, $ y_{hat} $)
    1. y, ground truth
    2. $ y_{hat} $, model's prediction
    1. E = y (ground truth) - $ h_{w(x)} $ [2]
    2. backward(): Back-Propagation is defined as first derive the gradient of each layer (or even each operation), then apply chain rule to compute the gradient of the whole network [1]
        1. We now discuss how to compute the gradient vector of the NLL by applying the chain rule of calculus. The resulting algorithm is known as backpropagation [5.1]
    3. criterion
        1. NLLLoss()
- epoch: Each cycle through the rnn with adjusting the weights slightly to reduce the error

References

1. CLASS: CSCI-544 Applied NLP by Xuezhe (Max) Ma, PhD @ USC
2. BOOK: Artificial Intelligence A Modern Appraoch | P 741
3. CLASS: CSCI-567 ML by Haipeng Luo, PhD @ USC
4. BOOK: Deep Learning by Ian Goodfellow, Yoshua Bengio, and Aaron Courville
    1. 4.1 P 151 - 153
5. BOOK: Machine Learning A Probabilistic Perspective by Kevin P. Murphy
    1. 5.1 P 570

### Code

In [ ]:
def train(classifier: nn.Module, sequence_of_embeddings, y, computer_loss, optimizer):
    """
    Train the RNN on a single sentence sequence.

    Parameters
    ----------
    classifier : nn.Module
        The recurrent neural network classifier.

    sequence_of_embeddings : list[np.ndarray] or list[torch.Tensor]
        Sequence of word embeddings for a single sentence.
        Example: [x_1, x_2, ..., x_T]

    y : torch.Tensor
        Ground truth label for the sentence.

    Returns
    -------
    final_output : torch.Tensor
        Final classifier prediction for the sequence.

    loss_value : float
        Loss value for the current sequence.
    """

    # 1. Initialize hidden state (h_0)
    hidden = classifier.resize_hidden()
    # print(f"Hidden: {hidden}")

    # 2. Iterate through sequence (t = 1 → T)
    for input_embedding_t in sequence_of_embeddings:
        x_embedding_t_reshaped = torch.tensor(input_embedding_t, dtype=torch.float32).unsqueeze(0)
        hidden, y_hat = classifier.forward(x_embedding_t_reshaped, hidden)
        # print(f"y_hat: {y_hat}")

    # 3. Final output = last timestep prediction
    final_output = y_hat

    # 4. Compute loss
    loss = computer_loss(final_output, y)
    # print(loss)

    # 5. Backpropagation
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)
    optimizer.step()

    # 6. Return results
    return final_output, loss.item()

#### Notes

- Loop through dataset row-by-row (each row = one word)
  
- If current word belongs to the same sentence:
  - Append word embedding to the current sequence  
  - Do **NOT** run RNN yet  

- If a new sentence is detected:
  - This marks the **end of the previous sentence**
  
  - Run RNN on the full sequence of word embeddings:
    - `[w1, w2, w3, ..., wT]`
  
  - Compute loss using the sentence-level ground truth  
  - Reset sequence  
  - Start collecting words for the next sentence  

- After loop ends:
  - Run RNN on the **final sequence** (last sentence)

---

> Mental Model

Each sentence is processed as a full sequence:

- Sentence 1:
  - `[w1, w2, w3] → RNN → loss`

- Sentence 2:
  - `[w4, w5] → RNN → loss`

- Sentence 3:
  - `[w6, w7, w8] → RNN → loss`

---

> ✅ Key Insight

- The RNN operates on:
  - **sequence of word embeddings per sentence**

- NOT:
  - individual words

#### Code

In [ ]:
# print(f"About to evaluate on test_embeddings_df with {len(test_embeddings_df)} rows")
# print(f"Unique sentences: {test_embeddings_df['Base Sentence'].nunique()}")
# print(test_embeddings_df.head())

print(f"test_embeddings_df has {len(test_embeddings_df)} rows")  # Should be ~238k
print(f"test_embeddings_df has {test_embeddings_df['Base Sentence'].nunique()} unique sentences")  # Should be 4962
test_embeddings_word_level_df = test_embeddings_df.copy()

In [ ]:
import copy

# Store initial weights
initial_W_hidden = rnn_classifier.input_to_hidden.weight.data.clone()
initial_W_out = rnn_classifier.hidden_to_output.weight.data.clone()

print("BEFORE TRAINING")
print(f"W_hidden[0,0] = {initial_W_hidden[0,0].item():.6f}")
print(f"W_out[0,0] = {initial_W_out[0,0].item():.6f}")
print()

criterion = nn.BCELoss()
learning_rate = 0.001
optimizer = torch.optim.Adam(rnn_classifier.parameters(), lr=learning_rate)

# Train on just FIRST 3 sentences
text_document_sequences = []
previous_text_document = None
current_y_tensor = None
sentence_count = 0

for row_idx, row_data in train_embeddings_df.iterrows():
    
    text_document = row_data['Base Sentence']
    word_embedding = row_data['Word Embedding']
    y = row_data['Ground Truth']
    y_tensor = torch.as_tensor([[y]], dtype=torch.float)
    
    if text_document == previous_text_document:
        text_document_sequences.append(word_embedding)
    else:
        if len(text_document_sequences) > 0:
            # Train on previous sentence
            output, loss = train(
                rnn_classifier,
                text_document_sequences,
                current_y_tensor,
                criterion,
                optimizer
            )
            
            sentence_count += 1
            
            # Show parameter changes after each sentence
            current_W_hidden = rnn_classifier.input_to_hidden.weight.data
            current_W_out = rnn_classifier.hidden_to_output.weight.data
            
            print(f"AFTER SENTENCE {sentence_count} ({text_document}):")
            print(f"  y_true: {current_y_tensor.item():.1f}")
            print(f"  y_hat: {output.squeeze().item():.4f}")
            print(f"  loss: {loss:.4f}")
            print(f"  W_hidden[0,0]: {current_W_hidden[0,0].item():.6f} (changed by {(current_W_hidden[0,0] - initial_W_hidden[0,0]).item():.6f})")
            print(f"  W_out[0,0]: {current_W_out[0,0].item():.6f} (changed by {(current_W_out[0,0] - initial_W_out[0,0]).item():.6f})")
            print()
            
            if sentence_count >= 3:
                break
        
        # Start new sequence
        text_document_sequences = [word_embedding]
        previous_text_document = text_document
        current_y_tensor = y_tensor

print("\nFINAL WEIGHTS (after 3 sentences):")
print(f"W_hidden[0,0] changed from {initial_W_hidden[0,0].item():.6f} to {current_W_hidden[0,0].item():.6f}")
print(f"W_out[0,0] changed from {initial_W_out[0,0].item():.6f} to {current_W_out[0,0].item():.6f}")

BEFORE TRAINING
W_hidden[0,0] = -0.214060
W_out[0,0] = 0.073736

AFTER SENTENCE 1 (Its shares closed at $5.125 each in composite New York Stock Exchange trading, down 37.5 cents.):
  y_true: 1.0
  y_hat: 1.0000
  loss: 0.0000
  W_hidden[0,0]: -0.214060 (changed by 0.000000)
  W_out[0,0]: 0.073736 (changed by 0.000000)

AFTER SENTENCE 2 (Cash flow from business operations totalled EUR 0.4 mn compared to a negative EUR 15.5 mn in the first half of 2008 .):
  y_true: 0.0
  y_hat: 0.0000
  loss: 0.0000
  W_hidden[0,0]: -0.214065 (changed by -0.000005)
  W_out[0,0]: 0.074467 (changed by 0.000731)

AFTER SENTENCE 3 (Will there be more than ten times as many 'Protests' in China for the 30 days before 2025-07-21 compared to one plus the 30-day average of 'Protests' over the 360 days preceding 2024-07-21?

e.g. If the forecast due date is 2024-01-01 and we have the following data:
Date,'Protests'
2023-11-11,1
2023-10-10,2
to calculate one plus the 30-day average of 'Protests' over the preceding

## Evaluate Model Performance

In [ ]:
# ✅ VERIFY which dataframe you're using
print(f"test_embeddings_df: {len(test_embeddings_df)} rows, {test_embeddings_df['Base Sentence'].nunique()} unique")
print(f"test_embeddings_df columns: {test_embeddings_df.columns.tolist()}")
print(f"Does test_embeddings_df have 'Word' column? {'Word' in test_embeddings_df.columns}")

In [ ]:
# Generate predictions for unique sentences
text_document_sequences = []
y_hats = []
previous_text_document = None

for row_idx in range(len(test_embeddings_df)):
    row = test_embeddings_df.iloc[row_idx]
    
    text_document = row['Base Sentence']
    word_embedding = row['Word Embedding']
    
    if text_document == previous_text_document:
        text_document_sequences.append(word_embedding)
    else:
        if len(text_document_sequences) > 0:
            with torch.no_grad():
                hidden = rnn_classifier.resize_hidden()
                for input_embedding_t in text_document_sequences:
                    x = torch.tensor(input_embedding_t, dtype=torch.float32).unsqueeze(0)
                    hidden, y_hat = rnn_classifier.forward(x, hidden)
                y_hats.append((y_hat.squeeze() > 0.5).long().item())
        
        text_document_sequences = [word_embedding]
        previous_text_document = text_document

# Handle last sequence
if len(text_document_sequences) > 0:
    with torch.no_grad():
        hidden = rnn_classifier.resize_hidden()
        for input_embedding_t in text_document_sequences:
            x = torch.tensor(input_embedding_t, dtype=torch.float32).unsqueeze(0)
            hidden, y_hat = rnn_classifier.forward(x, hidden)
        y_hats.append((y_hat.squeeze() > 0.5).long().item())

print(f"Generated {len(y_hats)} predictions for {test_embeddings_df['Base Sentence'].nunique()} unique sentences")

# ✅ Map predictions to original test_df (handles duplicates)
test_results_df = test_embeddings_df.groupby('Base Sentence', sort=False).first().reset_index()
sentence_to_prediction = dict(zip(test_results_df['Base Sentence'], y_hats))

# Apply to original test_df to preserve duplicates
test_final_df = test_df.copy()
test_final_df['RNN'] = test_final_df['Base Sentence'].map(sentence_to_prediction)

# Evaluate on full 5000 rows (with duplicates)
y = test_final_df['Ground Truth'].values
y_hat = test_final_df['RNN'].values

print(f"\nEvaluating on {len(test_final_df)} sentences (including {len(test_df) - len(y_hats)} duplicates)")
EvaluationMetric.eval_classification_report(y, y_hat)